In [1]:
import requests, json
import truststore; truststore.inject_into_ssl()     # แก้ SSLError จาก certificate chain ของ server

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
HEADERS = {"User-Agent": "pm25-explorer/0.1 (student project)"}

r = requests.get(URL, headers=HEADERS, timeout=(10, 60))
print(r.status_code, r.headers.get("Content-Type"), len(r.content), "bytes")
data = r.json()

# ถ้าดึงไม่ได้ ใช้ไฟล์ตัวอย่างของรายวิชา
# data = json.load(open("../data/sample/air4thai_sample.json", encoding="utf-8"))

200 application/json; charset=UTF-8 140111 bytes


In [2]:
data = r.json()
type(data)                        # dict
list(data.keys())                 # key ระดับบนสุด
st = data["stations"]
len(st), type(st[0])              # จำนวนสถานี, dict
list(st[0].keys())                # key ของแต่ละสถานี

['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast']

In [3]:
def show_keys(obj, prefix="", depth=3):
    """พิมพ์ path ของ key และชนิดค่า ไม่เกิน depth ชั้น"""
    if depth == 0:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            print(f"{prefix}{k}: {type(v).__name__}")
            show_keys(v, prefix + "  ", depth - 1)
    elif isinstance(obj, list) and obj:
        show_keys(obj[0], prefix + "[0] ", depth - 1)

show_keys(data["stations"][0])

stationID: str
nameTH: str
nameEN: str
areaTH: str
areaEN: str
stationType: str
lat: str
long: str
forecast: list
AQILast: dict
  date: str
  time: str
  PM25: dict
    color_id: str
    aqi: str
    value: str
  PM10: dict
    color_id: str
    aqi: str
    value: str
  O3: dict
    color_id: str
    aqi: str
    value: str
  CO: dict
    color_id: str
    aqi: str
    value: str
  NO2: dict
    color_id: str
    aqi: str
    value: str
  SO2: dict
    color_id: str
    aqi: str
    value: str
  AQI: dict
    color_id: str
    aqi: str
    param: str


| คอลัมน์ในตารางของเรา | path ของ key ใน JSON | ชนิดใน JSON | ตัวอย่างค่า |
| :--- | :--- | :--- | :--- |
| `station_id` | `stationID` | `str` | `"119t"` |
| `name_th` / `name_en` | `nameTH` / `nameEN` | `str` / `str` | `"สวนสาธารณะธารา"` / `"Thara Public Park"` |
| `province` (อยู่ในข้อความพื้นที่?) | `areaTH` / `areaEN` (ไม่มี field `province` โดยตรง ต้องตัดคำจากข้อความ เช่น ข้อความหลังจุลภาค `,` ใน `areaTH`) | `str` | `"ต.ปากน้ำ อ.เมือง, กระบี่"` (สกัดได้ `"กระบี่"`) / `"Pak Nam Subdistrict, Mueang District, Krabi"` (สกัดได้ `"Krabi"`) |
| `lat` / `lon` | `lat` / `long` (ข้อสังเกต: ใน JSON ใช้ชื่อ key ว่า `long` ไม่ใช่ `lon`) | `str` / `str` (เก็บเป็นสตริงตัวเลข ต้องแปลงเป็น `FLOAT`) | `"8.0506237"` / `"98.9180489"` |
| `station_type` | `stationType` | `str` | `"GROUND"` (หรือ `"MOBILE"`) |
| วันที่ + เวลาของค่าล่าสุด | `AQILast.date` และ `AQILast.time` (นำมาต่อกันเป็น datetime) | `str` (ทั้งสองค่า) | `"2026-11-18"` (date), `"00:00"` (time) รวมเป็น `"2026-11-18 00:00"` |
| `pm25` (ความเข้มข้น ไม่ใช่ AQI) | `AQILast.PM25.value` (ระวัง: ไม่ใช่ `AQILast.PM25.aqi`) | `str` (เก็บเป็นสตริงตัวเลข ต้องแปลงเป็น `FLOAT`) | `"6.3"` (หรือ `"-1"` หากไม่มีข้อมูล) |
| `pm10`, `o3`, `co`, `no2`, `so2` | `AQILast.PM10.value`<br>`AQILast.O3.value`<br>`AQILast.CO.value`<br>`AQILast.NO2.value`<br>`AQILast.SO2.value` | `str` (เก็บเป็นสตริงตัวเลขทั้งหมด ต้องแปลงเป็น `FLOAT`) | `PM10.value`: `"-1"`<br>`O3.value`: `"-1"`<br>`CO.value`: `"-1"`<br>`NO2.value`: `"-1"`<br>`SO2.value`: `"-1"` |

---

### ตอบคำถามเพิ่มเติม

1. **ค่าหายถูกแทนด้วยอะไร?**
   - **ค่าความเข้มข้นสารมลพิษ (`value`)**: ถูกแทนด้วยสตริงตัวเลขติดลบคือ `"-1"` (เช่น `"PM10": {"value": "-1", ...}`)
   - **ค่าดัชนี (`aqi`)**: ถูกแทนด้วยสตริง `"-1"` หรือ `"-999"` (และ `color_id` จะเป็น `"0"`)
   - **ข้อควรระวังสำหรับ `load_duckdb.py`**: ค่าทั้งหมดใน JSON ส่งมาเป็น **String (`str`)** ไม่ใช่ `null` ของ JSON ดังนั้นเมื่อโหลดลง DuckDB ต้องแปลงชนิดข้อมูล (`TRY_CAST(... AS FLOAT)`) และแปลงค่า `"-1"` หรือค่าน้อยกว่า 0 ให้กลายเป็น `NULL` เพื่อไม่ให้ค่า -1 ถูกนำไปคำนวณทางสถิติและทำให้ค่าเฉลี่ยผิดเพี้ยน

2. **สถานีทุกสถานีมี key ครบเท่ากันหรือไม่?**
   - **มี key ครบเท่ากันทุกสถานี 100% (Uniform Schema)** จากการตรวจสอบทั้ง 178 สถานี:
     - ระดับ Root ของสถานีมี 10 keys ครบเท่ากันทั้งหมด: `stationID`, `nameTH`, `nameEN`, `areaTH`, `areaEN`, `stationType`, `lat`, `long`, `forecast`, `AQILast`
     - ระดับ `AQILast` มี 9 keys ครบเท่ากันทั้งหมด: `date`, `time`, `PM25`, `PM10`, `O3`, `CO`, `NO2`, `SO2`, `AQI`
     - ระดับมลพิษย่อย (`PM25`, `PM10`, `O3`, `CO`, `NO2`, `SO2`) มี 3 keys ครบเท่ากันทั้งหมด: `color_id`, `aqi`, `value`
     - ระดับ `AQI` มี 3 keys ครบเท่ากันทั้งหมด: `color_id`, `aqi`, `param`
     - **สรุป**: โครงสร้าง Schema มีความสม่ำเสมอ แม้สถานีนั้นจะไม่มีการวัดสารมลพิษชนิดนั้น ตัว API จะไม่ตัดคีย์ทิ้ง แต่ยังส่งคีย์มาครบและแทนค่า `value` ด้วย `"-1"` แทน

---

In [4]:
from collections import Counter
Counter(tuple(sorted(s.keys())) for s in st).most_common(3)   # รูปแบบ key ที่พบบ่อย

[(('AQILast',
   'areaEN',
   'areaTH',
   'forecast',
   'lat',
   'long',
   'nameEN',
   'nameTH',
   'stationID',
   'stationType'),
  180)]

In [5]:
from urllib.robotparser import RobotFileParser
rp = RobotFileParser("http://air4thai.pcd.go.th/robots.txt")
rp.read()
rp.can_fetch(HEADERS["User-Agent"], URL)

True